In [1]:
import requests as re
response=re.get('https://books.toscrape.com')
htmt=response.text

In [2]:
htmt

'<!DOCTYPE html>\n<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->\n<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->\n<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->\n<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->\n    <head>\n        <title>\n    All products | Books to Scrape - Sandbox\n</title>\n\n        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />\n        <meta name="created" content="24th Jun 2016 09:29" />\n        <meta name="description" content="" />\n        <meta name="viewport" content="width=device-width" />\n        <meta name="robots" content="NOARCHIVE,NOCACHE" />\n\n        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->\n        <!--[if lt IE 9]>\n        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>\n        <![endif]-->\n\n        \n            <link rel="shortcut icon" href

In [23]:
import requests as re
from bs4 import BeautifulSoup as bs
page = re.get('https://books.toscrape.com')
def main(page):
    src=page.content
    soup = bs(src, 'lxml')
    livres=soup.find_all('article',class_="product_pod")
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }
    for livre in livres:
        titre=livre.find('h3').find('a')['title']
        prix=livre.find('p',class_='price_color').text
        rating_class = livre.find('p', class_='star-rating')['class']
        rating_text = rating_class[1]
        rating_num = rating_map.get(rating_text, 0)
        print(titre,prix,rating_num)
main(page)


A Light in the Attic £51.77 3
Tipping the Velvet £53.74 1
Soumission £50.10 1
Sharp Objects £47.82 4
Sapiens: A Brief History of Humankind £54.23 5
The Requiem Red £22.65 1
The Dirty Little Secrets of Getting Your Dream Job £33.34 4
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull £17.93 3
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics £22.60 4
The Black Maria £52.15 1
Starving Hearts (Triangular Trade Trilogy, #1) £13.99 2
Shakespeare's Sonnets £20.66 4
Set Me Free £17.46 5
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) £52.29 5
Rip it Up and Start Again £35.02 5
Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 £57.25 3
Olio £23.88 1
Mesaerion: The Best Science Fiction Stories 1800-1849 £37.59 1
Libertarianism for Beginners £51.33 2
It's Only the Himalayas £45.17 2


Prender tout les livres

In [24]:
import requests as re
from bs4 import BeautifulSoup as bs

BASE_URL = "https://books.toscrape.com/"

def get_books_from_page(url):
    page = re.get(url)
    soup = bs(page.content, "lxml")
    livres = soup.find_all("article", class_="product_pod")

    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    books = []
    for livre in livres:
        titre = livre.find("h3").find("a")["title"]
        prix = livre.find("p", class_="price_color").text
        rating_class = livre.find("p", class_="star-rating")["class"]
        rating_text = rating_class[1]
        rating_num = rating_map.get(rating_text, 0)

        books.append((titre, prix, rating_num))
    return books, soup

def main():
    url = BASE_URL
    while True:
        books, soup = get_books_from_page(url)
        for titre, prix, rating in books:
            print(titre, prix, rating)

        # pagination : vérifier s’il y a un "next"
        next_btn = soup.find("li", class_="next")
        if next_btn:
            next_page = next_btn.find("a")["href"]
            # construire l’URL complète
            if "catalogue/" not in url and "catalogue/" in next_page:
                url = BASE_URL + next_page
            else:
                url = BASE_URL + "catalogue/" + next_page
        else:
            break

main()


A Light in the Attic £51.77 3
Tipping the Velvet £53.74 1
Soumission £50.10 1
Sharp Objects £47.82 4
Sapiens: A Brief History of Humankind £54.23 5
The Requiem Red £22.65 1
The Dirty Little Secrets of Getting Your Dream Job £33.34 4
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull £17.93 3
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics £22.60 4
The Black Maria £52.15 1
Starving Hearts (Triangular Trade Trilogy, #1) £13.99 2
Shakespeare's Sonnets £20.66 4
Set Me Free £17.46 5
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) £52.29 5
Rip it Up and Start Again £35.02 5
Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 £57.25 3
Olio £23.88 1
Mesaerion: The Best Science Fiction Stories 1800-1849 £37.59 1
Libertarianism for Beginners £51.33 2
It's Only the Himalayas £45.17 2
In Her Wake £12.84 1
How Music Works £37.32 2
Foolproof Preserving: A Guide to Small Batch Ja

Récupérer l’URL de la page de chaque livre.

Visiter cette page → extraire description + stock.

Les ajouter au dictionnaire avant d’enregistrer dans le CSV.

In [25]:
import requests as re
from bs4 import BeautifulSoup as bs
import csv

BASE_URL = "https://books.toscrape.com/"

def get_book_details(book_url):
    """Récupère la description et le stock depuis la page d’un livre"""
    page = re.get(book_url)
    soup = bs(page.content, "lxml")

    # description
    desc = soup.find("div", id="product_description")
    if desc:
        description = desc.find_next_sibling("p").text
    else:
        description = "No description available"

    # stock
    stock_text = soup.find("p", class_="instock availability").text.strip()
    # Exemple : "In stock (22 available)" → garder juste 22
    stock_num = "".join([c for c in stock_text if c.isdigit()]) or "0"

    return description, stock_num

def get_books_from_page(url):
    page = re.get(url)
    soup = bs(page.content, "lxml")
    livres = soup.find_all("article", class_="product_pod")

    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    books = []
    for livre in livres:
        titre = livre.find("h3").find("a")["title"]
        prix = livre.find("p", class_="price_color").text
        rating_class = livre.find("p", class_="star-rating")["class"]
        rating_text = rating_class[1]
        rating_num = rating_map.get(rating_text, 0)

        # lien relatif du livre
        link = livre.find("h3").find("a")["href"]
        if not link.startswith("http"):
            if "catalogue/" not in link:
                book_url = BASE_URL + "catalogue/" + link
            else:
                book_url = BASE_URL + link
        else:
            book_url = link

        # récupérer description + stock
        description, stock = get_book_details(book_url)

        # stocker les infos
        books.append({
            "Titre": titre,
            "Prix": prix,
            "Rating": rating_num,
            "Stock": stock,
            "Description": description
        })
    return books, soup

def main():
    url = BASE_URL
    all_books = []

    while True:
        books, soup = get_books_from_page(url)
        all_books.extend(books)

        next_btn = soup.find("li", class_="next")
        if next_btn:
            next_page = next_btn.find("a")["href"]
            if "catalogue/" not in url and "catalogue/" in next_page:
                url = BASE_URL + next_page
            else:
                url = BASE_URL + "catalogue/" + next_page
        else:
            break

    # Sauvegarder en CSV
    with open("books.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["Titre", "Prix", "Rating", "Stock", "Description"])
        writer.writeheader()
        writer.writerows(all_books)

    print("✅ Données sauvegardées dans books.csv")

main()


✅ Données sauvegardées dans books.csv
